# VoiceSecure 훈련 노트북

**실행 순서대로 셀을 실행하세요.**

### 사전 준비
- 런타임 유형을 **GPU (T4 이상)** 로 설정하세요. (런타임 > 런타임 유형 변경)
- Google Drive에 KSS 데이터셋 폴더를 업로드해두세요.

### 파이프라인 개요
```
원본 음성
  └─▶ RLAgent.act()          — state 추출 (MFCC+F0+RMS, 16-dim) → 노이즈 action (257×100)
  └─▶ PsychoacousticMasker   — 심리음향 임계치로 noise clamp (사람에게 안 들리는 범위)
  └─▶ Mixer.mix()            — 원본 + safe_noise → 변조 음성
  └─▶ Reward 계산
         ├─ 평소 (9/10):    ECAPA dist(원본, 변조) × 0.5 + CAM++ dist × 0.5
         └─ TTS 평가 (1/10): CosyVoice3 클로닝 후 clone dist (실제 방어 점수)
  └─▶ PPO update             — 32 transition마다
```

## Step 1. GPU 확인

In [ ]:
import torch, sys

print('Python:', sys.version)
print('PyTorch:', torch.__version__)
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('WARNING: GPU 없음 — 런타임 > 런타임 유형 변경 > GPU 설정 필요')

## Step 2. Google Drive 마운트

아래 셀 실행 후 Drive 연결 허용을 눌러주세요.  
`KSS_DIR`을 본인 Drive의 KSS 경로로 수정하세요.

예) Drive에 `내 드라이브/kss` 폴더 → `/content/drive/MyDrive/kss`

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# ↓ 본인 Drive의 KSS 경로로 수정
KSS_DIR = '/content/drive/MyDrive/kss'
print('KSS_DIR:', KSS_DIR)

## Step 3. CosyVoice 레포 클론

CosyVoice3 TTS 모델 사용에 필요합니다.  
이미 `/content/CosyVoice`가 있으면 스킵합니다.

In [ ]:
import os

COSYVOICE_ROOT = '/content/CosyVoice'

if not os.path.exists(COSYVOICE_ROOT):
    !git clone --recursive https://github.com/FunAudioLLM/CosyVoice.git {COSYVOICE_ROOT}
    print('CosyVoice 클론 완료')
else:
    print('CosyVoice 이미 존재, 스킵')

## Step 4. 의존성 설치

설치 패키지:
- `CosyVoice/requirements.txt` — CosyVoice3 실행에 필요한 전체 의존성
- `speechbrain` — ECAPA-TDNN 화자 임베딩 모델
- `soundfile` — 오디오 파일 I/O (librosa 대신 사용, k2 충돌 방지)
- `scipy` — 리샘플링 (resample_poly)
- `tensorboard` — 훈련 로그 시각화

> **주의**: speechbrain 설치 후 k2 관련 경고가 나올 수 있으나 무시해도 됩니다.  
> 이 프로젝트는 librosa를 사용하지 않아 k2 충돌이 발생하지 않습니다.

In [ ]:
!pip install -r {COSYVOICE_ROOT}/requirements.txt -q
!pip install speechbrain soundfile scipy tensorboard -q
print('의존성 설치 완료')

## Step 5. CosyVoice3 모델 다운로드

HuggingFace에서 `Fun-CosyVoice3-0.5B-2512` 모델을 다운로드합니다.  
약 **3~5 GB**, 최초 실행 시 수 분 소요됩니다.

In [ ]:
MODEL_DIR = f'{COSYVOICE_ROOT}/pretrained_models/Fun-CosyVoice3-0.5B-2512'

if not os.path.exists(MODEL_DIR):
    from huggingface_hub import snapshot_download
    snapshot_download(
        repo_id='FunAudioLLM/Fun-CosyVoice3-0.5B-2512',
        local_dir=MODEL_DIR,
    )
    print('모델 다운로드 완료:', MODEL_DIR)
else:
    print('모델 이미 존재, 스킵:', MODEL_DIR)

## Step 6. VoiceSecure SDK 설치

GitHub에서 SDK를 클론하고 editable 모드로 설치합니다.  
이미 있으면 최신 코드로 업데이트(`git pull`)합니다.

In [ ]:
SDK_ROOT = '/content/voicesecure-sdk'

if not os.path.exists(SDK_ROOT):
    !git clone https://github.com/VoiceSecureHoseo/voicesecure-sdk.git {SDK_ROOT}
else:
    !git -C {SDK_ROOT} pull
    print('SDK 업데이트 완료')

%cd {SDK_ROOT}
!pip install -e . -q
print('SDK 설치 완료')

## Step 7. 설치 검증

SDK 임포트, ECAPA-TDNN 로드, CosyVoice3 로드를 순서대로 확인합니다.

> **ECAPA-TDNN**: SpeechBrain이 HuggingFace에서 모델 weights를 자동 다운로드합니다.  
> 최초 실행 시 1~2분 소요됩니다.

In [ ]:
import sys

# CosyVoice Python path 추가 (import를 위해 필요)
for p in [COSYVOICE_ROOT, f'{COSYVOICE_ROOT}/third_party/Matcha-TTS']:
    if p not in sys.path:
        sys.path.insert(0, p)

# SDK 핵심 모듈 임포트
from voicesecure.modulation.masker import PsychoacousticMasker
from voicesecure.modulation.mixer import Mixer
from voicesecure.rl.agent import RLAgent
from voicesecure.types import ACTION_N_FREQ, ACTION_N_TIME, SAMPLE_RATE
print(f'SDK 임포트 OK  (action shape: {ACTION_N_FREQ}×{ACTION_N_TIME})')

# ECAPA-TDNN 로드 (최초 실행 시 HF에서 weights 다운로드)
from voicesecure.evaluators.adapters.ecapa_tdnn import ECAPATDNNAdapter
ecapa = ECAPATDNNAdapter()   # device 자동 감지 (CUDA 우선)
print(f'ECAPA-TDNN OK  (device={ecapa.device})')

# CosyVoice3 로드
from voicesecure.evaluators.adapters.cosyvoice import CosyVoiceAdapter
cosy = CosyVoiceAdapter(model_dir=MODEL_DIR, cosyvoice_root=COSYVOICE_ROOT)
print('CosyVoice3 OK')

## Step 8. 빠른 동작 확인 (1 에피소드)

훈련 전 파이프라인 전체가 정상 동작하는지 확인합니다.

- 합성 오디오(1초, 16kHz)로 act → clamp → mix → reward 한 사이클을 돌립니다.

In [ ]:
import numpy as np
from voicesecure.evaluators.base import cosine_distance

# 합성 테스트 오디오 (1초)
rng = np.random.default_rng(42)
t = np.arange(SAMPLE_RATE, dtype=np.float32) / SAMPLE_RATE
test_audio = (
    0.3 * np.sin(2 * np.pi * 200 * t)
    + 0.2 * np.sin(2 * np.pi * 800 * t)
    + 0.02 * rng.standard_normal(SAMPLE_RATE).astype(np.float32)
).astype(np.float32)

# 모듈 초기화
agent  = RLAgent()
masker = PsychoacousticMasker()
mixer  = Mixer()

# 1 에피소드 실행
orig_ecapa = ecapa.extract_embedding(test_audio)
orig_cam   = cosy.extract_embedding(test_audio)

state, action, log_prob, value, freq_pattern, time_gate = agent.act(test_audio)
safe_noise = masker.clamp(test_audio, action)
modified   = mixer.mix(test_audio, safe_noise)

mod_ecapa = ecapa.extract_embedding(modified)
mod_cam   = cosy.extract_embedding(modified)

ecapa_dist = cosine_distance(orig_ecapa, mod_ecapa)
cam_dist   = cosine_distance(orig_cam, mod_cam)
reward     = 0.5 * ecapa_dist + 0.5 * cam_dist

print(f'action shape : {tuple(action.shape)}  (기대: (257, 100))')
print(f'modified shape: {modified.shape}  dtype={modified.dtype}')
print(f'ECAPA dist   : {ecapa_dist:.4f}')
print(f'CAM++ dist   : {cam_dist:.4f}')
print(f'reward       : {reward:.4f}')
print('파이프라인 정상 동작 확인')

## Step 9. 훈련 실행

### 훈련 하이퍼파라미터
| 파라미터 | 기본값 | 설명 |
|---|---|---|
| `--epochs` | 3 | 1 에폭 = KSS 전체 파일 수 에피소드 |
| `--lr` | 3e-4 | PPO Adam learning rate |
| `--checkpoint_interval` | 1000 | 체크포인트 저장 주기 (에피소드) |
| `--resume` | None | 이전 체크포인트 경로 (이어서 훈련) |

### Reward 구조
- **평소 (매 9 에피소드)**: `ECAPA_dist(원본, 변조) × 0.5 + CAM++_dist × 0.5`  
  → 임베딩 공간에서 원본과 변조 음성이 얼마나 멀어졌는지
- **TTS 평가 (10 에피소드마다)**: CosyVoice3로 실제 클로닝 후 `clone_dist`  
  → 실제 공격 시나리오 기준 방어 효과, PPO 방향을 교정

### TensorBoard
- `train/reward`, `train/ecapa_dist`, `train/cam_dist` — 에피소드별
- `eval/tts_defense`, `eval/tts_ecapa_dist`, `eval/tts_cam_dist` — TTS 평가 에피소드
- `train/policy_loss`, `train/value_loss`, `train/entropy` — PPO 업데이트마다

In [ ]:
CHECKPOINT_DIR = f'{SDK_ROOT}/checkpoints'

!python {SDK_ROOT}/train.py \
    --data_dir       {KSS_DIR} \
    --model_dir      {MODEL_DIR} \
    --cosyvoice_root {COSYVOICE_ROOT} \
    --epochs         3 \
    --checkpoint_dir {CHECKPOINT_DIR}

### 이어서 훈련 (런타임 재연결 후)

Colab 런타임이 끊겼다가 재연결 시 아래 셀로 이어서 훈련할 수 있습니다.  
`RESUME_CKPT`를 재개할 체크포인트 경로로 수정하세요.

In [ ]:
# ↓ 재개할 체크포인트 경로 수정 (best.pt 또는 episode_N.pt)
RESUME_CKPT = f'{CHECKPOINT_DIR}/best.pt'

!python {SDK_ROOT}/train.py \
    --data_dir       {KSS_DIR} \
    --model_dir      {MODEL_DIR} \
    --cosyvoice_root {COSYVOICE_ROOT} \
    --epochs         3 \
    --checkpoint_dir {CHECKPOINT_DIR} \
    --resume         {RESUME_CKPT}

## Step 10. TensorBoard 로그 확인

In [ ]:
%load_ext tensorboard
%tensorboard --logdir {CHECKPOINT_DIR}/logs

## Step 11. 방어 효과 평가

훈련된 에이전트로 KSS 파일 하나를 변조하고,  
CosyVoice3 클로닝 전/후 ECAPA 거리를 비교합니다.

**목표 지표**: cosine distance > 0.3 (SIM < 0.25 논문 기준)

In [ ]:
import soundfile as sf
from math import gcd
from pathlib import Path
from scipy.signal import resample_poly
from voicesecure.evaluators.base import cosine_distance

def load_audio(path, sr=16000):
    data, orig_sr = sf.read(str(path), dtype='float32', always_2d=True)
    audio = data.mean(axis=1).astype(np.float32)
    if orig_sr != sr:
        g = gcd(sr, orig_sr)
        audio = resample_poly(audio, sr // g, orig_sr // g).astype(np.float32)
    return np.clip(audio, -1.0, 1.0)

# 체크포인트 로드
agent_eval = RLAgent()
agent_eval.load(f'{CHECKPOINT_DIR}/best.pt')

# 평가할 파일 선택 (KSS 첫 번째 wav)
test_wav = next(Path(KSS_DIR).rglob('*.wav'))
test_text = '안녕하세요'  # ↓ 실제 텍스트로 수정하면 더 정확
print('평가 파일:', test_wav)

original = load_audio(test_wav)

# 원본 임베딩
orig_ecapa = ecapa.extract_embedding(original)

# 변조
_, action, _, _, _, _ = agent_eval.act(original, deterministic=True)
safe_noise = masker.clamp(original, action)
modified   = mixer.mix(original, safe_noise)

# 변조 음성 직접 임베딩 거리
mod_ecapa = ecapa.extract_embedding(modified)
direct_dist = cosine_distance(orig_ecapa, mod_ecapa)

# 클로닝 후 거리 (실제 공격 시나리오)
cloned      = cosy.clone(modified, text=test_text)
clone_ecapa = ecapa.extract_embedding(cloned)
clone_dist  = cosine_distance(orig_ecapa, clone_ecapa)

print(f'\n--- 방어 효과 평가 ---')
print(f'직접 ECAPA 거리 (원본 vs 변조):  {direct_dist:.4f}')
print(f'클론 ECAPA 거리 (원본 vs 클론):  {clone_dist:.4f}  ← 핵심 지표')
print(f'목표:                            > 0.30')
print(f'달성 여부:                       {"성공" if clone_dist > 0.30 else "미달"}')

## Step 12. 체크포인트 Drive 백업

Colab 런타임 종료 시 `/content` 안의 파일은 삭제됩니다.  
훈련이 끝나면 아래 셀로 Drive에 백업하세요.

In [ ]:
import shutil

# ↓ Drive 백업 경로 수정
BACKUP_DIR = '/content/drive/MyDrive/voicesecure_checkpoints'

shutil.copytree(CHECKPOINT_DIR, BACKUP_DIR, dirs_exist_ok=True)
print(f'백업 완료: {BACKUP_DIR}')